In [2]:
from transformers import pipeline
from datasets import load_dataset
import evaluate
import time
from PIL import Image
import pandas as pd

# Task 1

In [3]:
# Sentiment Analysis Pipeline
sentiment_model_A = pipeline(
    'text-classification', 
    model='lvwerra/distilbert-imdb'
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [4]:
# Test both to show they work
content = "Darwin will love the new CSM movie, but will hate the ending of GOT"
output_A = sentiment_model_A(content)
print(output_A)
# Model is 62% confident that the sentence is positive rather than negative

[{'label': 'POSITIVE', 'score': 0.6275621056556702}]


In [5]:
ner_pipeline = pipeline(
    "ner", 
    model="sartajbhuvaji/bert-named-entity-recognition", 
    aggregation_strategy="simple"
)
content_ner = "Youssef says that the new CSM movie is very good, so you might want to check it out"
ner_pipeline(content_ner)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[{'entity_group': 'PER',
  'score': np.float32(0.9989951),
  'word': 'you',
  'start': 0,
  'end': 3},
 {'entity_group': 'PER',
  'score': np.float32(0.81375265),
  'word': '##ssef',
  'start': 3,
  'end': 7},
 {'entity_group': 'MISC',
  'score': np.float32(0.8366738),
  'word': 'cs',
  'start': 26,
  'end': 28},
 {'entity_group': 'ORG',
  'score': np.float32(0.7085727),
  'word': '##m',
  'start': 28,
  'end': 29}]

# Task 2

In [6]:
sentiment_model_B = pipeline(
    'text-classification', 
    model='siebert/sentiment-roberta-large-english'
)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: siebert/sentiment-roberta-large-english
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
output_B = sentiment_model_B(content)
print(output_B)


[{'label': 'NEGATIVE', 'score': 0.9658474326133728}]


In [19]:
def evaluate_model(pipe, inputs, expected_labels, model_name, accuracy_metric, f1_metric):
    start = time.time()
    # Run inference with batches for faster runtime (process more text at a time)
    predictions = list(pipe(inputs, batch_size=16, truncation=True)) # truncate input if too long
    total_time = time.time() - start
    average_time = total_time / len(inputs)
    
    # Format predictions
    pred_labels = [1 if "pos" in pred['label'].lower() else 0 for pred in predictions]
    
    # Calculate Metrics
    acc = accuracy_metric.compute(predictions=pred_labels, references=expected_labels)['accuracy']
    f1 = f1_metric.compute(predictions=pred_labels, references=expected_labels)['f1']
    
    return {
        "Model": model_name,
        "Accuracy": round(acc, 4),
        "F1 Score": round(f1, 4),
        "Avg Latency (s)": round(average_time, 4)
    }

In [14]:
# Load 200 examples of the IMDB test set
full_dataset = load_dataset('imdb', split='test')
shuffled_dataset = full_dataset.shuffle(seed=42).select(range(200))
texts = list(shuffled_dataset['text']) # a dataset of reviews
true_labels = shuffled_dataset['label'] # 0 = Negative, 1 = Positive

# Load evaluation metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

In [15]:
texts[0]

"<br /><br />When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story and of course Michelle Pfeiffer was in it, so what could go wrong?<br /><br />Very quickly, however, I realized that this story was about A Thousand Other Things besides just Acres. I started crying and couldn't stop until long after the movie ended. Thank you Jane, Laura and Jocelyn, for bringing us such a wonderfully subtle and compassionate movie! Thank you cast, for being involved and portraying the characters with such depth and gentleness!<br /><br />I recognized the Angry sister; the Runaway sister and the sister in Denial. I recognized the Abusive Husband and why he was there and then the Father, oh oh the Father... all superbly played. I also recognized myself and this movie was an eye-opener, a relief, a chance to face my OWN truth and finally doing something about it. I truly hope A Thousand Acres has had the same effect on some others out there.<br /><br />Since

In [16]:
true_labels

Column([1, 1, 0, 1, 0, ...])

In [20]:
results_a = evaluate_model(sentiment_model_A, texts, true_labels, "DistilBERT IMDB", accuracy_metric, f1_metric)
results_b = evaluate_model(sentiment_model_B, texts, true_labels, "Roberta", accuracy_metric, f1_metric)

In [21]:
result = pd.DataFrame([results_a, results_b])
display(result)

,Model,Accuracy,F1 Score,Avg Latency (s)
0,DistilBERT IMDB,0.92,0.9167,0.0039
1,Roberta,0.93,0.9255,0.0200
